In [23]:
# Cell 1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import re

from datetime import datetime
import warnings
import os
import chardet

warnings.filterwarnings("ignore")

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

class OptimizedFeatureEngineer:
    """优化版特征工程类 - 基于086.txt的思路大幅改进"""
    
    def __init__(self):
        self.feature_stats = {}
        self.bin_edges = {}
        
    def load_data_smart(self, file_path):
        """智能加载数据"""
        try:
            with open(file_path, 'rb') as f:
                encoding = chardet.detect(f.read())['encoding']
            
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
            print(f"✅ 成功加载: {file_path}, 形状: {df.shape}")
            return df
        except Exception as e:
            print(f"❌ 加载失败 {file_path}: {e}")
            return None
    
    def parse_build_year(self, s):
        """解析建筑年代"""
        if pd.isnull(s): return np.nan
        s = str(s); years = re.findall(r'(\d{4})', s)
        if not years: return np.nan
        return np.mean([float(y) for y in years]) if len(years) > 1 else float(years[0])
    
    def parse_first_number(self, s, default_val=np.nan):
        """解析第一个数字"""
        if pd.isnull(s): return default_val
        s = str(s).replace('暂无', '0')
        match = re.search(r'(\d+\.?\d*)', s)
        return float(match.group(1)) if match else default_val
    
    def extract_advanced_features(self, df):
        """高级特征工程 - 基于086.txt的改进版本"""
        df_feat = df.copy()
        
        print("🔧 开始高级特征工程...")
        
        # 中文数字映射
        chinese_num_map = {
            "一": "1", "二": "2", "两": "2", "三": "3", "四": "4", "五": "5", 
            "六": "6", "七": "7", "八": "8", "九": "9", "十": "10"
        }
        
        # 1. 户型解析
        if '房屋户型' in df_feat.columns:
            try:
                # 计算众数用于填充
                huxing_mode = df_feat['房屋户型'].mode()[0] if not df_feat['房屋户型'].mode().empty else '2室1厅1卫'
                temp_huxing = df_feat['房屋户型'].fillna(huxing_mode)
                
                shi_series = temp_huxing.str.extract(r'(\d+)室', expand=False).astype(float)
                shi_mode = shi_series.mode().iloc[0] if not shi_series.mode().empty else 2.0
                
                ting_series = temp_huxing.str.extract(r'(\d+)厅', expand=False).astype(float)
                ting_mode = ting_series.mode().iloc[0] if not ting_series.mode().empty else 1.0
                
                wei_series = temp_huxing.str.extract(r'(\d+)卫', expand=False).astype(float)
                wei_mode = wei_series.mode().iloc[0] if not wei_series.mode().empty else 1.0
                
                df_feat['室'] = df_feat['房屋户型'].str.extract(r'(\d+)室', expand=False).astype(float).fillna(shi_mode)
                df_feat['厅'] = df_feat['房屋户型'].str.extract(r'(\d+)厅', expand=False).astype(float).fillna(ting_mode)
                df_feat['卫'] = df_feat['房屋户型'].str.extract(r'(\d+)卫', expand=False).astype(float).fillna(wei_mode)
                df_feat['房间总数'] = df_feat['室'] + df_feat['厅'] + df_feat['卫']
                df_feat['卫室比'] = df_feat['卫'] / (df_feat['室'] + 0.01)
                
            except Exception as e:
                print(f"⚠️ 户型解析错误: {e}")
        
        # 2. 楼层信息解析
        if '所在楼层' in df_feat.columns:
            try:
                floor_str = df_feat['所在楼层'].astype(str)
                df_feat['总楼层'] = floor_str.str.extract(r'共(\d+)层')[0].astype(float)
                
                # 楼层位置映射
                floor_position_str = floor_str.str.extract(r'(地下室|底层|顶层|低楼层|中楼层|高楼层)')[0]
                floor_position_map = {'地下室': -1, '底层': 1, '低楼层': 2, '中楼层': 3, '高楼层': 4, '顶层': 5}
                df_feat['楼层位置'] = floor_position_str.map(floor_position_map).fillna(3)
                
                # 计算楼层相对位置
                df_feat['总楼层'] = df_feat['总楼层'].clip(lower=1)
                df_feat['楼层相对位置'] = df_feat['楼层位置'] / df_feat['总楼层']
                
            except Exception as e:
                print(f"⚠️ 楼层信息解析错误: {e}")
        
        # 3. 面积特征
        if '建筑面积' in df_feat.columns:
            try:
                df_feat['建筑面积_数值'] = df_feat['建筑面积'].astype(str).str.replace('㎡', '').astype(float)
                df_feat['面积_log'] = np.log1p(df_feat['建筑面积_数值'])
                # 分箱处理
                _, edges = pd.qcut(df_feat['建筑面积_数值'].dropna(), q=6, labels=False, retbins=True, duplicates='drop')
                self.bin_edges['建筑面积_数值'] = np.unique(edges)
                df_feat['面积分箱'] = pd.cut(df_feat['建筑面积_数值'], bins=self.bin_edges['建筑面积_数值'], labels=False, include_lowest=True).fillna(-1)
                
            except Exception as e:
                print(f"⚠️ 面积解析错误: {e}")
        
        # 4. 建筑年代和房龄
        if '建筑年代' in df_feat.columns:
            try:
                df_feat['建筑年份'] = df_feat['建筑年代'].apply(self.parse_build_year)
                df_feat['房龄'] = 2025 - df_feat['建筑年份'].fillna(2000)
                df_feat['是否新房'] = (df_feat['房龄'] <= 5).astype(int)
                # 房龄分箱
                _, edges = pd.qcut(df_feat['房龄'].dropna(), q=6, labels=False, retbins=True, duplicates='drop')
                self.bin_edges['房龄'] = np.unique(edges)
                df_feat['房龄分箱'] = pd.cut(df_feat['房龄'], bins=self.bin_edges['房龄'], labels=False, include_lowest=True).fillna(-1)
                
            except Exception as e:
                print(f"⚠️ 建筑年代解析错误: {e}")
        
        # 5. 朝向特征
        if '房屋朝向' in df_feat.columns:
            try:
                directions = ['东', '南', '西', '北']
                for direction in directions:
                    df_feat[f'朝{direction}'] = df_feat['房屋朝向'].astype(str).str.contains(direction, na=False).astype(int)
                df_feat['南北通透'] = ((df_feat['朝南'] == 1) & (df_feat['朝北'] == 1)).astype(int)
                df_feat['朝南权重'] = df_feat['朝南'] * 2 + df_feat['朝东'] * 1 + df_feat['朝北'] * 0.5 - df_feat['朝西'] * 0.5
                
            except Exception as e:
                print(f"⚠️ 朝向解析错误: {e}")
        
        # 6. 装修等级
        if '装修情况' in df_feat.columns:
            try:
                renovation_map = {'毛坯': 1, '简装': 2, '精装': 3, '其他': 2}
                df_feat['装修等级'] = df_feat['装修情况'].map(renovation_map).fillna(2)
            except Exception as e:
                print(f"⚠️ 装修等级解析错误: {e}")
        
        # 7. 梯户比例解析
        if '梯户比例' in df_feat.columns:
            try:
                temp_col = df_feat['梯户比例'].astype(str)
                sorted_keys = sorted(chinese_num_map.keys(), key=len, reverse=True)
                for char in sorted_keys: 
                    num_str = chinese_num_map[char]
                    temp_col = temp_col.str.replace(char, num_str)
                
                df_feat['梯数'] = temp_col.str.extract(r'(\d+)梯').astype(float).fillna(1)
                df_feat['户数'] = temp_col.str.extract(r'(\d+)户').astype(float).fillna(2)
                df_feat['户数'] = df_feat['户数'].replace(0, 2)
                df_feat['梯户比'] = df_feat['梯数'] / df_feat['户数']
                
                # 盖帽处理
                p_99_hu = df_feat['户数'].quantile(0.99)
                p_99_ti = df_feat['梯数'].quantile(0.99)
                df_feat['户数'] = df_feat['户数'].clip(upper=min(p_99_hu, 50))
                df_feat['梯数'] = df_feat['梯数'].clip(upper=min(p_99_ti, 20))
                df_feat['梯户比'] = df_feat['梯户比'].clip(upper=df_feat['梯户比'].quantile(0.99))
                
            except Exception as e:
                print(f"⚠️ 梯户比例解析错误: {e}")
        
        # 8. 水电特征重新编码
        if '供水' in df_feat.columns:
            df_feat['供水_商水'] = df_feat['供水'].str.contains('商水', na=False).astype(int)
            df_feat['供水_民水'] = df_feat['供水'].str.contains('民水', na=False).astype(int)
        
        if '供电' in df_feat.columns:
            df_feat['供电_商电'] = df_feat['供电'].str.contains('商电', na=False).astype(int)
            df_feat['供电_民电'] = df_feat['供电'].str.contains('民电', na=False).astype(int)
        
        # 9. 配备电梯映射
        if '配备电梯' in df_feat.columns:
            df_feat['配备电梯_mapped'] = df_feat['配备电梯'].map({'有': 1, '无': 0}).fillna(0).astype(int)
        
        # 10. 其他数值特征解析
        numeric_features_mapping = {
            '房屋总数': ('房屋总数_num', '户'),
            '楼栋总数': ('楼栋总数_num', '栋'), 
            '绿 化 率': ('绿化率_num', '%'),
            '容 积 率': ('容积率_num', None),
            '燃气费': ('燃气费_num', None),
            '停车位': ('停车位_num', None),
            '停车费用': ('停车费用_num', None)
        }
        
        for orig_col, (new_col, unit) in numeric_features_mapping.items():
            if orig_col in df_feat.columns:
                try:
                    if unit:
                        df_feat[new_col] = pd.to_numeric(
                            df_feat[orig_col].astype(str).str.replace(unit, '', regex=False), 
                            errors='coerce'
                        )
                    else:
                        if orig_col == '燃气费':
                            df_feat[new_col] = df_feat[orig_col].apply(self.parse_first_number)
                        else:
                            df_feat[new_col] = pd.to_numeric(df_feat[orig_col], errors='coerce')
                    
                    # 特殊处理
                    if orig_col == '绿 化 率':
                        df_feat[new_col] = df_feat[new_col] / 100.0
                        df_feat[new_col] = df_feat[new_col].clip(upper=1.0)
                    elif orig_col == '容 积 率':
                        df_feat[new_col] = df_feat[new_col].clip(upper=df_feat[new_col].quantile(0.99))
                        
                except Exception as e:
                    print(f"⚠️ {orig_col} 解析错误: {e}")
        
        # 11. 日期特征
        if '交易时间' in df_feat.columns:
            df_feat['交易时间'] = pd.to_datetime(df_feat['交易时间'], errors='coerce')
            df_feat['交易年份'] = df_feat['交易时间'].dt.year
            df_feat['交易月份'] = df_feat['交易时间'].dt.month
        
        if '上次交易' in df_feat.columns:
            df_feat['上次交易'] = pd.to_datetime(df_feat['上次交易'], errors='coerce')
            if '交易时间' in df_feat.columns:
                df_feat['持有天数'] = (df_feat['交易时间'] - df_feat['上次交易']).dt.days
                df_feat['持有天数'] = df_feat['持有天数'].fillna(0).clip(lower=0)
                df_feat['是否首次交易'] = df_feat['上次交易'].isnull().astype(int)
        
        # 12. 文本特征标志
        text_cols = ['房屋优势', '周边配套', '交通出行', '核心卖点', '客户反馈']
        for col in text_cols:
            if col in df_feat.columns:
                df_feat[f'有_{col}'] = df_feat[col].notnull().astype(int)
                if col == '核心卖点':
                    df_feat[f'{col}_长度'] = df_feat[col].str.len().fillna(0)
        
        # 13. 多标签二值化
        if '建筑结构' in df_feat.columns:
            main_types_jiegou = ['塔楼', '板楼', '塔板结合', '平房']
            for j_type in main_types_jiegou:
                df_feat[f'建筑结构_{j_type}'] = df_feat['建筑结构'].astype(str).str.contains(j_type, na=False).astype(int)
        
        if '物业类别' in df_feat.columns:
            main_types = ['普通住宅', '别墅', '写字楼', '商业', '公寓', '底商']
            for prop_type in main_types:
                df_feat[f'物业类型_{prop_type}'] = df_feat['物业类别'].astype(str).str.contains(prop_type, na=False).astype(int)
        
        if '房屋用途' in df_feat.columns:
            main_types_yongtu = ['普通住宅', '别墅', '商业办公类', '车库', '公寓', '住宅']
            for y_type in main_types_yongtu:
                df_feat[f'房屋用途_{y_type}'] = df_feat['房屋用途'].astype(str).str.contains(y_type, na=False).astype(int)
        
        print(f"✅ 特征工程完成，创建了 {len([col for col in df_feat.columns if col not in df.columns])} 个新特征")
        
        return df_feat
    
    def remove_data_leakage_features(self, df):
        """移除数据泄露特征"""
        leakage_patterns = [
            '均价', '平均价', '评估价', '预测价', 'estimated', 'predicted', 
            '小区均价', 'community', 'neighborhood', 'score', 'rating', 'rank'
        ]
        
        leakage_cols = []
        for col in df.columns:
            col_lower = str(col).lower()
            if any(pattern in col_lower for pattern in leakage_patterns):
                leakage_cols.append(col)
        
        if leakage_cols:
            print(f"🚫 移除数据泄露特征: {leakage_cols}")
            df = df.drop(columns=leakage_cols, errors='ignore')
        
        return df
    
    def efficient_missing_value_imputation(self, df, target_col=None):
        """高效缺失值填充"""
        df_clean = df.copy()
        
        print("🔧 处理缺失值...")
        
        # 1. 移除高缺失率特征
        missing_rates = df_clean.isnull().mean()
        high_missing_cols = missing_rates[missing_rates > 0.7].index.tolist()
        if high_missing_cols:
            print(f"🚫 移除高缺失率特征: {high_missing_cols}")
            df_clean = df_clean.drop(columns=high_missing_cols)
        
        # 2. 关键特征不能缺失 - 删除缺失行
        critical_cols = ['建筑面积_数值', '总楼层', '楼层位置']
        critical_cols_exist = [col for col in critical_cols if col in df_clean.columns]
        
        if critical_cols_exist:
            before = len(df_clean)
            df_clean = df_clean.dropna(subset=critical_cols_exist)
            after = len(df_clean)
            print(f"🧹 移除关键特征缺失行: {before - after} 行")
        
        # 3. 快速填充数值列
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        if target_col and target_col in numeric_cols:
            numeric_cols = numeric_cols.drop(target_col)
        
        for col in numeric_cols:
            if df_clean[col].isnull().any():
                # 根据偏态选择填充策略
                if df_clean[col].skew() > 2:
                    fill_value = df_clean[col].median()
                else:
                    fill_value = df_clean[col].mean()
                df_clean[col].fillna(fill_value, inplace=True)
        
        # 4. 分类列快速填充
        categorical_cols = df_clean.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df_clean[col].isnull().any():
                mode_val = df_clean[col].mode()
                fill_value = mode_val[0] if not mode_val.empty else '未知'
                df_clean[col].fillna(fill_value, inplace=True)
        
        print("✅ 缺失值处理完成")
        
        return df_clean
    
    def remove_outliers_iqr(self, df, target_col):
        """移除异常值"""
        if target_col not in df.columns:
            print(f"⚠️ 目标列 {target_col} 不存在，跳过异常值处理")
            return df
        
        try:
            # 使用更保守的5%和95%分位数
            Q1 = df[target_col].quantile(0.05)
            Q3 = df[target_col].quantile(0.95)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 2 * IQR  # 使用2倍IQR，更宽松
            upper_bound = Q3 + 2 * IQR
            
            before = len(df)
            df_clean = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)]
            after = len(df_clean)
            
            print(f"✅ 异常值处理: 移除 {before - after} 个样本, 剩余 {after} 个样本")
            
            return df_clean
        except Exception as e:
            print(f"⚠️ 异常值处理错误: {e}")
            return df

In [24]:
# Cell 2
class SmartDataProcessor:
    """智能数据处理类 - 基于086.txt思路大幅改进"""
    
    def __init__(self):
        self.feature_engineer = OptimizedFeatureEngineer()
        self.imputation_strategies = {}
        self.outlier_bounds = {}
        self.scaler = RobustScaler()
        
    def load_and_detect_encoding(self, file_path):
        """智能加载数据并检测编码"""
        try:
            with open(file_path, 'rb') as f:
                encoding = chardet.detect(f.read())['encoding']
            
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
            print(f"✅ 成功加载: {file_path} (编码: {encoding}, 形状: {df.shape})")
            return df
        except Exception as e:
            print(f"❌ 加载失败 {file_path}: {e}")
            return None
    
    def combine_datasets(self, train_path, test_path, target_col='Price'):
        """合并训练和测试集"""
        train_df = self.load_and_detect_encoding(train_path)
        test_df = self.load_and_detect_encoding(test_path)
        
        if train_df is None or test_df is None:
            return None, None, None
        
        # 添加标识列
        train_df['is_train'] = 1
        test_df['is_train'] = 0
        
        # 合并数据
        combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
        
        print(f"📊 数据合并完成: 训练集 {train_df.shape}, 测试集 {test_df.shape}, 合并 {combined_df.shape}")
        return combined_df, train_df, test_df
    
    def comprehensive_feature_engineering(self, df):
        """综合特征工程"""
        print("🔧 开始综合特征工程...")
        
        # 1. 移除数据泄露特征
        df = self.feature_engineer.remove_data_leakage_features(df)
        
        # 2. 高级特征工程
        df = self.feature_engineer.extract_advanced_features(df)
        
        # 3. 空间聚类特征
        df = self.create_spatial_clusters(df)
        
        print(f"✅ 特征工程完成: 形状 {df.shape}")
        return df
    
    def create_spatial_clusters(self, df):
        """创建空间聚类特征"""
        coord_cols = []
        if 'lon' in df.columns and 'lat' in df.columns:
            coord_cols = ['lon', 'lat']
        elif 'coord_x' in df.columns and 'coord_y' in df.columns:
            coord_cols = ['coord_x', 'coord_y']
        
        if coord_cols and df[coord_cols].notna().all().all():
            try:
                # 使用肘部法则确定最优聚类数量
                coords = df[coord_cols].dropna().values
                if len(coords) > 10:  # 确保有足够的数据点
                    k_range = range(3, 8)
                    wcss = []
                    
                    for k in k_range:
                        kmeans = KMeans(n_clusters=k, random_state=111, n_init=10)
                        kmeans.fit(coords)
                        wcss.append(kmeans.inertia_)
                    
                    # 计算斜率变化找到肘部
                    if len(wcss) > 2:
                        wcss_diff = [wcss[i-1] - wcss[i] for i in range(1, len(wcss))]
                        wcss_diff_diff = [wcss_diff[i-1] - wcss_diff[i] for i in range(1, len(wcss_diff))]
                        optimal_k = k_range[wcss_diff_diff.index(max(wcss_diff_diff)) + 2] if wcss_diff_diff else 5
                    else:
                        optimal_k = 5
                    
                    # 应用聚类
                    kmeans_final = KMeans(n_clusters=optimal_k, random_state=111, n_init=10)
                    df.loc[df[coord_cols].notna().all(axis=1), 'location_cluster'] = kmeans_final.fit_predict(
                        df.loc[df[coord_cols].notna().all(axis=1), coord_cols]
                    )
                    print(f"✅ 空间聚类完成: {optimal_k} 个聚类")
            except Exception as e:
                print(f"⚠️ 空间聚类错误: {e}")
        
        return df
    
    def smart_data_cleaning(self, combined_df, target_col='Price'):
        """智能数据清洗流程"""
        print("🧹 开始智能数据清洗...")
        
        # 分离训练测试集
        train_df = combined_df[combined_df['is_train'] == 1].copy()
        test_df = combined_df[combined_df['is_train'] == 0].copy()
        
        # 训练集专用处理
        if target_col in train_df.columns:
            # 移除目标变量缺失值
            before = len(train_df)
            train_df = train_df.dropna(subset=[target_col])
            after = len(train_df)
            print(f"🧹 移除目标缺失样本: {before - after} 行")
            
            # 异常值处理（只对训练集）
            train_df = self.feature_engineer.remove_outliers_iqr(train_df, target_col)
        
        # 重新合并
        df_clean = pd.concat([train_df, test_df], axis=0)
        
        # 缺失值填充
        df_clean = self.feature_engineer.efficient_missing_value_imputation(df_clean, target_col)
        
        print("✅ 数据清洗完成")
        return df_clean
    
    def prepare_final_features(self, combined_df, target_col='Price'):
        """准备最终特征"""
        print("🎯 准备最终特征...")
        
        # 分离训练测试集
        train_df = combined_df[combined_df['is_train'] == 1].copy()
        test_df = combined_df[combined_df['is_train'] == 0].copy()
        
        print(f"训练集: {train_df.shape}, 测试集: {test_df.shape}")
        
        # 移除标识列
        train_df = train_df.drop('is_train', axis=1, errors='ignore')
        test_df = test_df.drop('is_train', axis=1, errors='ignore')
        
        # 目标变量处理（只对训练集）
        if target_col in train_df.columns:
            # 移除目标变量缺失值
            train_df = train_df.dropna(subset=[target_col])
            
            # 对数转换（处理偏态）
            y_train = np.log1p(train_df[target_col])
            train_df = train_df.drop([target_col], axis=1)
        else:
            y_train = None
        
        # 移除测试集中的目标列
        if target_col in test_df.columns:
            test_df = test_df.drop(target_col, axis=1)
        
        # 选择数值型特征
        numeric_cols_train = train_df.select_dtypes(include=[np.number]).columns
        numeric_cols_test = test_df.select_dtypes(include=[np.number]).columns
        common_numeric_cols = numeric_cols_train.intersection(numeric_cols_test)
        
        X_train = train_df[common_numeric_cols]
        X_test = test_df[common_numeric_cols]
        
        # 最终清理
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
        
        # 移除方差为零的特征
        variance = X_train.var()
        zero_variance_cols = variance[variance == 0].index
        if len(zero_variance_cols) > 0:
            print(f"移除零方差特征: {list(zero_variance_cols)}")
            X_train = X_train.drop(columns=zero_variance_cols, errors='ignore')
            X_test = X_test.drop(columns=zero_variance_cols, errors='ignore')
        
        # 移除高度相关特征
        X_train, X_test = self.remove_highly_correlated_features(X_train, X_test)
        
        print(f"✅ 最终特征: X_train {X_train.shape}, X_test {X_test.shape}")
        print(f"📈 特征数量: {X_train.shape[1]}")
        
        return X_train, X_test, y_train
    
    def remove_highly_correlated_features(self, X_train, X_test, threshold=0.95):
        """移除高度相关特征"""
        try:
            # 计算相关系数矩阵
            corr_matrix = X_train.corr().abs()
            
            # 选择上三角矩阵
            upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
            
            # 找到相关性高于阈值的特征
            to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
            
            if to_drop:
                print(f"移除高度相关特征: {to_drop}")
                X_train = X_train.drop(columns=to_drop, errors='ignore')
                X_test = X_test.drop(columns=to_drop, errors='ignore')
            
            return X_train, X_test
        except Exception as e:
            print(f"⚠️ 移除相关特征错误: {e}")
            return X_train, X_test

In [25]:
# Cell 3: 基于Untitled5非线性经验改进的高级模型训练器
from sklearn.metrics import make_scorer
from scipy import stats
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LassoCV

class AdvancedModelTrainer:
    """基于Untitled5非线性经验大幅改进的高级模型训练器"""
    
    def __init__(self, property_type='price'):
        self.property_type = property_type
        self.models = {}
        self.best_model = None
        self.best_model_name = None
        self.feature_importance = None
        self.cv_results = {}
        self.important_features = None
        self.nonlinear_insights = {}  # 存储从非线性模型学到的经验
        
    def extract_nonlinear_insights(self, X_train, y_train):
        """从非线性模型中提取关键经验 - 基于Untitled5的成功经验"""
        print("🔍 从非线性模型提取关键经验...")
        
        # 使用随机森林快速识别真正重要的特征（仅用于指导线性模型）
        rf_insight = RandomForestRegressor(n_estimators=100, random_state=111, n_jobs=-1)
        rf_insight.fit(X_train, y_train)
        
        # 获取特征重要性
        importance_df = pd.DataFrame({
            'feature': X_train.columns,
            'importance': rf_insight.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # 基于Untitled5经验：非线性模型发现的关键特征模式
        self.nonlinear_insights = {
            'top_features': importance_df.head(30)['feature'].tolist(),
            'feature_importance': importance_df,
            'interaction_terms': self.identify_potential_interactions(importance_df, X_train),
            'nonlinear_patterns': self.analyze_nonlinear_patterns(X_train, y_train, importance_df)
        }
        
        print(f"✅ 提取到 {len(self.nonlinear_insights['top_features'])} 个关键特征")
        print(f"📊 前5重要特征: {self.nonlinear_insights['top_features'][:5]}")
        
        return self.nonlinear_insights
    
    def identify_potential_interactions(self, importance_df, X_train):
        """识别潜在的特征交互作用 - 基于非线性模型的经验"""
        top_10_features = importance_df.head(10)['feature'].tolist()
        interactions = []
        
        # 基于领域知识和特征重要性创建交互项
        area_related = [f for f in top_10_features if '面积' in f or 'room' in f.lower()]
        location_related = [f for f in top_10_features if 'lon' in f or 'lat' in f or '城市' in f or '区域' in f]
        
        # 创建有意义的交互项组合
        for area_feat in area_related[:3]:
            for loc_feat in location_related[:3]:
                interactions.append(f"{area_feat}_x_{loc_feat}")
        
        return interactions
    
    def analyze_nonlinear_patterns(self, X_train, y_train, importance_df):
        """分析非线性模式 - 指导线性模型改进"""
        patterns = {}
        top_features = importance_df.head(15)['feature'].tolist()
        
        for feature in top_features[:5]:  # 只分析前5个最重要特征
            if feature in X_train.columns:
                # 分析特征与目标的关系
                correlation = np.corrcoef(X_train[feature], y_train)[0,1]
                patterns[feature] = {
                    'correlation': correlation,
                    'suggested_transform': self.suggest_transformation(X_train[feature], y_train)
                }
        
        return patterns
    
    def suggest_transformation(self, feature_series, target_series):
        """基于数据分布建议特征变换"""
        skewness = stats.skew(feature_series.dropna())
        
        if abs(skewness) > 1.0:
            return 'log' if skewness > 0 else 'square_root'
        elif abs(skewness) > 0.5:
            return 'polynomial'
        else:
            return 'original'
    
    def create_advanced_pipeline(self, model, use_feature_selection=True, use_polynomial=True, use_insights=True):
        """创建基于非线性经验的高级预处理管道"""
        steps = [
            ('scaler', RobustScaler()),  # 使用RobustScaler处理异常值
        ]
        
        # 基于非线性经验：多项式特征可以捕捉非线性关系
        if use_polynomial and use_insights:
            # 更智能的多项式特征，基于非线性模型学到的模式
            steps.append(('poly', PolynomialFeatures(
                degree=2, 
                include_bias=False, 
                interaction_only=True  # 只保留交互项，避免特征爆炸
            )))
        
        # 基于非线性经验：特征选择很重要
        if use_feature_selection and use_insights:
            # 使用基于非线性模型学到的特征重要性进行选择
            steps.append(('feature_selection', SelectKBest(score_func=f_regression, k=25)))
        
        steps.append(('model', model))
        
        return Pipeline(steps)
    
    def apply_nonlinear_insights(self, X_train, X_test=None):
        """应用从非线性模型学到的经验"""
        if not self.nonlinear_insights:
            return X_train, X_test
        
        X_train_enhanced = X_train.copy()
        if X_test is not None:
            X_test_enhanced = X_test.copy()
        
        # 应用特征变换建议
        for feature, insights in self.nonlinear_insights.get('nonlinear_patterns', {}).items():
            if feature in X_train_enhanced.columns:
                transform_type = insights.get('suggested_transform', 'original')
                
                if transform_type == 'log' and (X_train_enhanced[feature] > 0).all():
                    X_train_enhanced[f'{feature}_log'] = np.log1p(X_train_enhanced[feature])
                    if X_test is not None and (X_test_enhanced[feature] > 0).all():
                        X_test_enhanced[f'{feature}_log'] = np.log1p(X_test_enhanced[feature])
                
                elif transform_type == 'square_root' and (X_train_enhanced[feature] >= 0).all():
                    X_train_enhanced[f'{feature}_sqrt'] = np.sqrt(X_train_enhanced[feature])
                    if X_test is not None and (X_test_enhanced[feature] >= 0).all():
                        X_test_enhanced[f'{feature}_sqrt'] = np.sqrt(X_test_enhanced[feature])
        
        # 添加交互项
        for interaction in self.nonlinear_insights.get('interaction_terms', []):
            parts = interaction.split('_x_')
            if len(parts) == 2 and all(part in X_train_enhanced.columns for part in parts):
                X_train_enhanced[interaction] = X_train_enhanced[parts[0]] * X_train_enhanced[parts[1]]
                if X_test is not None and all(part in X_test_enhanced.columns for part in parts):
                    X_test_enhanced[interaction] = X_test_enhanced[parts[0]] * X_test_enhanced[parts[1]]
        
        if X_test is not None:
            return X_train_enhanced, X_test_enhanced
        return X_train_enhanced
    
    def create_custom_scorer(self):
        """创建自定义评分器（在原始价格尺度）"""
        def original_price_mae(y_log, y_pred_log):
            y_orig = np.expm1(y_log)
            y_pred_orig = np.expm1(y_pred_log)
            
            # 处理极端值
            y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, 
                                      posinf=np.finfo(np.float64).max, 
                                      neginf=0.0)
            y_pred_orig = np.clip(y_pred_orig, 0, None)
            
            return mean_absolute_error(y_orig, y_pred_orig)
        
        return make_scorer(original_price_mae, greater_is_better=False)
    
    def calculate_kaggle_score(self, mae, property_type):
        """计算Kaggle风格分数 - 基于Untitled5的基准调整"""
        if property_type == 'price':
            # 基于Untitled5经验：随机森林达到了20万MAE，调整基准
            base_mae = 300000  # 比随机森林稍宽松的基准
            score = max(0, 100 * (1 - mae / base_mae))
        else:
            # 房租基准调整为2000（基于Untitled5的表现）
            base_mae = 2000
            score = max(0, 100 * (1 - mae / base_mae))
        
        return score
    
    def train_linear_models_with_insights(self, X_train, y_train, X_val=None, y_val=None):
        """基于非线性经验训练改进的线性模型"""
        print("📈 基于非线性经验训练改进的线性模型...")
        
        # 首先从非线性模型提取经验
        self.extract_nonlinear_insights(X_train, y_train)
        
        # 应用非线性经验增强特征
        if X_val is not None:
            X_train_enhanced, X_val_enhanced = self.apply_nonlinear_insights(X_train, X_val)
        else:
            X_train_enhanced = self.apply_nonlinear_insights(X_train)
            X_val_enhanced = None
        
        results = []
        mae_scorer = self.create_custom_scorer()
        
        # 使用非线性模型识别的重要特征
        important_features = self.nonlinear_insights['top_features']
        X_train_important = X_train_enhanced[important_features]
        X_val_important = X_val_enhanced[important_features] if X_val_enhanced is not None else None
        
        # 基于Untitled5经验优化的模型配置
        models_config = {
            'OLS_Enhanced': {
                'model': LinearRegression(),
                'params': {},
                'use_polynomial': True,
                'use_feature_selection': False,
                'use_important_features': True
            },
            'Ridge_Enhanced': {
                'model': Ridge(random_state=111),
                'params': {
                    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
                },
                'use_polynomial': True,
                'use_feature_selection': True,
                'use_important_features': True
            },
            'Lasso_Enhanced': {
                'model': Lasso(random_state=111, max_iter=10000),
                'params': {
                    'model__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0]
                },
                'use_polynomial': False,  # Lasso本身有特征选择
                'use_feature_selection': False,
                'use_important_features': True
            },
            'ElasticNet_Enhanced': {
                'model': ElasticNet(random_state=111, max_iter=10000),
                'params': {
                    'model__alpha': [0.001, 0.01, 0.1],
                    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
                },
                'use_polynomial': False,
                'use_feature_selection': False,
                'use_important_features': True
            }
        }
        
        for name, config in models_config.items():
            try:
                # 创建基于非线性经验的管道
                pipeline = self.create_advanced_pipeline(
                    config['model'], 
                    use_feature_selection=config['use_feature_selection'],
                    use_polynomial=config['use_polynomial'],
                    use_insights=True
                )
                
                # 选择特征集
                if config['use_important_features']:
                    X_tr = X_train_important
                    X_vl = X_val_important
                else:
                    X_tr = X_train_enhanced
                    X_vl = X_val_enhanced
                
                if config['params']:
                    # 使用交叉验证调参
                    cv_folds = min(5, max(3, len(X_tr) // 100))
                    grid_search = GridSearchCV(
                        pipeline,
                        config['params'],
                        cv=cv_folds,
                        scoring=mae_scorer,
                        n_jobs=-1,
                        verbose=0
                    )
                    
                    grid_search.fit(X_tr, y_train)
                    best_model = grid_search.best_estimator_
                    best_params = grid_search.best_params_
                    best_score = -grid_search.best_score_
                else:
                    # 直接训练
                    best_model = pipeline
                    best_model.fit(X_tr, y_train)
                    best_params = {}
                    best_score = None
                
                # 存储模型
                self.models[name] = best_model
                
                # 计算训练集指标
                y_train_pred = best_model.predict(X_tr)
                mae_train = mean_absolute_error(np.expm1(y_train), np.expm1(y_train_pred))
                r2_train = r2_score(y_train, y_train_pred)
                
                # 计算验证集指标
                if X_vl is not None and y_val is not None:
                    y_val_pred = best_model.predict(X_vl)
                    mae_val = mean_absolute_error(np.expm1(y_val), np.expm1(y_val_pred))
                    r2_val = r2_score(y_val, y_val_pred)
                else:
                    mae_val = mae_train
                    r2_val = r2_train
                
                # 交叉验证分数
                if best_score is None:
                    cv_scores = cross_validate(
                        best_model, X_tr, y_train,
                        cv=min(5, max(3, len(X_tr) // 100)),
                        scoring=mae_scorer,
                        n_jobs=-1
                    )
                    mae_cv = -np.mean(cv_scores['test_score'])
                else:
                    mae_cv = best_score
                
                # Kaggle分数
                kaggle_score = self.calculate_kaggle_score(mae_val, self.property_type)
                
                result = {
                    'Model': name,
                    'In_sample_MAE': mae_train,
                    'Out_sample_MAE': mae_val,
                    'CV_MAE': mae_cv,
                    'R2_Train': r2_train,
                    'R2_Val': r2_val,
                    'Kaggle_Score': kaggle_score,
                    'Best_Params': best_params,
                    'Features_Used': X_tr.shape[1],
                    'Nonlinear_Insights_Applied': True
                }
                results.append(result)
                
                print(f"✅ {name}: 训练MAE = {mae_train:,.2f}, 验证MAE = {mae_val:,.2f}")
                
            except Exception as e:
                print(f"❌ {name} 训练失败: {e}")
                continue
        
        return pd.DataFrame(results)
    
    def select_best_model(self, results):
        """选择最佳模型"""
        if results.empty:
            print("❌ 没有成功的模型训练")
            return None
        
        # 选择验证集MAE最小的模型
        best_idx = results['Out_sample_MAE'].idxmin()
        self.best_model_name = results.loc[best_idx, 'Model']
        self.best_model = self.models.get(self.best_model_name)
        
        print(f"\n🏆 最佳模型: {self.best_model_name}")
        print(f"📊 验证集MAE: {results.loc[best_idx, 'Out_sample_MAE']:,.2f}")
        print(f"⭐ Kaggle分数: {results.loc[best_idx, 'Kaggle_Score']:.1f}")
        
        return results
    
    def predict_with_insights(self, X):
        """使用基于非线性经验的模型预测"""
        if self.best_model is None:
            raise ValueError("❌ 没有可用的训练模型")
        
        # 应用相同的特征增强
        X_enhanced = self.apply_nonlinear_insights(X)
        
        # 如果使用了重要特征，选择相同的特征
        if self.nonlinear_insights and 'top_features' in self.nonlinear_insights:
            important_features = self.nonlinear_insights['top_features']
            # 确保测试集包含所有重要特征
            missing_features = [f for f in important_features if f not in X_enhanced.columns]
            for feature in missing_features:
                X_enhanced[feature] = 0  # 填充默认值
            
            X_final = X_enhanced[important_features]
        else:
            X_final = X_enhanced
        
        y_pred_log = self.best_model.predict(X_final)
        y_pred = np.expm1(y_pred_log)
        
        # 后处理 - 基于Untitled5的经验
        y_pred = np.clip(y_pred, 0, None)
        y_pred = np.nan_to_num(y_pred, nan=np.median(y_pred))
        
        # 基于Untitled5经验：对极端值进行更智能的平滑处理
        q95 = np.percentile(y_pred, 95)
        q5 = np.percentile(y_pred, 5)
        y_pred = np.where(y_pred > q95 * 1.5, q95 * 1.2, y_pred)
        y_pred = np.where(y_pred < q5 * 0.5, q5 * 0.8, y_pred)
        
        return y_pred

In [26]:
# Cell 4: 模型评估和可视化
class ModelEvaluator:
    """模型评估和可视化类"""
    
    def __init__(self):
        self.results = {}
        
    def create_comparison_plot(self, results_dict):
        """创建模型比较图"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 准备数据
        all_results = []
        for prop_type, results in results_dict.items():
            if not results.empty:
                results_copy = results.copy()
                results_copy['Property_Type'] = prop_type
                all_results.append(results_copy)
        
        if not all_results:
            print("❌ 没有结果数据可展示")
            return
        
        comparison_df = pd.concat(all_results, ignore_index=True)
        
        # 1. MAE比较
        models = comparison_df['Model'].unique()
        x_pos = np.arange(len(models))
        
        for i, prop_type in enumerate(results_dict.keys()):
            prop_data = comparison_df[comparison_df['Property_Type'] == prop_type]
            if prop_data.empty:
                continue
                
            # 按模型名称排序以确保一致性
            prop_data = prop_data.set_index('Model').reindex(models).reset_index()
            
            axes[0, i].bar(x_pos - 0.2, prop_data['In_sample_MAE'], 0.4, 
                          label='训练集', alpha=0.7)
            axes[0, i].bar(x_pos + 0.2, prop_data['Out_sample_MAE'], 0.4, 
                          label='验证集', alpha=0.7)
            axes[0, i].set_title(f'{prop_type} - MAE比较')
            axes[0, i].set_xlabel('模型')
            axes[0, i].set_ylabel('MAE')
            axes[0, i].set_xticks(x_pos)
            axes[0, i].set_xticklabels(models, rotation=45)
            axes[0, i].legend()
            axes[0, i].grid(True, alpha=0.3)
        
        # 2. Kaggle分数比较
        for i, prop_type in enumerate(results_dict.keys()):
            prop_data = comparison_df[comparison_df['Property_Type'] == prop_type]
            if prop_data.empty:
                continue
                
            prop_data = prop_data.set_index('Model').reindex(models).reset_index()
            
            axes[1, i].bar(x_pos, prop_data['Kaggle_Score'], alpha=0.7, color='green')
            axes[1, i].set_title(f'{prop_type} - Kaggle分数')
            axes[1, i].set_xlabel('模型')
            axes[1, i].set_ylabel('分数')
            axes[1, i].set_xticks(x_pos)
            axes[1, i].set_xticklabels(models, rotation=45)
            axes[1, i].grid(True, alpha=0.3)
            
            # 在柱子上添加数值
            for j, score in enumerate(prop_data['Kaggle_Score']):
                axes[1, i].text(j, score + 1, f'{score:.1f}', 
                               ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        return comparison_df
    
    def plot_residual_analysis(self, model, X, y, model_name):
        """绘制残差分析图"""
        y_pred_log = model.predict(X)
        y_true = np.expm1(y)
        y_pred = np.expm1(y_pred_log)
        
        residuals = y_true - y_pred
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. 残差分布
        axes[0, 0].hist(residuals, bins=50, alpha=0.7, edgecolor='black')
        axes[0, 0].axvline(0, color='red', linestyle='--')
        axes[0, 0].set_title(f'{model_name} - 残差分布')
        axes[0, 0].set_xlabel('残差')
        axes[0, 0].set_ylabel('频数')
        
        # 2. 残差vs预测值
        axes[0, 1].scatter(y_pred, residuals, alpha=0.5)
        axes[0, 1].axhline(0, color='red', linestyle='--')
        axes[0, 1].set_title(f'{model_name} - 残差vs预测值')
        axes[0, 1].set_xlabel('预测值')
        axes[0, 1].set_ylabel('残差')
        
        # 3. 预测值vs真实值
        axes[1, 0].scatter(y_true, y_pred, alpha=0.5)
        max_val = max(y_true.max(), y_pred.max())
        axes[1, 0].plot([0, max_val], [0, max_val], 'r--')
        axes[1, 0].set_title(f'{model_name} - 预测值vs真实值')
        axes[1, 0].set_xlabel('真实值')
        axes[1, 0].set_ylabel('预测值')
        
        # 4. QQ图
        stats.probplot(residuals, dist="norm", plot=axes[1, 1])
        axes[1, 1].set_title(f'{model_name} - QQ图')
        
        plt.tight_layout()
        plt.show()
        
        # 打印统计信息
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        print(f"📊 {model_name} 残差分析:")
        print(f"   MAE: {mae:,.2f}")
        print(f"   R²: {r2:.4f}")
        print(f"   残差均值: {residuals.mean():.2f}")
        print(f"   残差标准差: {residuals.std():.2f}")

In [27]:
# Cell 5: 基于非线性经验改进的主流程
def run_enhanced_real_estate_pipeline():
    """运行基于非线性经验改进的房地产预测流程"""
    print("=== 🏠 基于Untitled5非线性经验改进的房地产预测系统 ===\n")
    
    # 文件路径
    price_train_path = "ruc_Class25Q2_train_price.csv"
    price_test_path = "ruc_Class25Q2_test_price.csv"
    rent_train_path = "ruc_Class25Q2_train_rent.csv" 
    rent_test_path = "ruc_Class25Q2_test_rent.csv"
    
    # 初始化组件
    processor = SmartDataProcessor()
    all_predictions = []
    all_results = {}
    
    # ===== 房价预测 =====
    print("\n" + "="*60)
    print("🏘️  房价建模（基于非线性经验优化）")
    print("="*60)
    
    # 加载和预处理数据
    combined_price, price_train, price_test = processor.combine_datasets(
        price_train_path, price_test_path, 'Price'
    )
    
    if combined_price is not None:
        # 数据预处理流程
        combined_price = processor.comprehensive_feature_engineering(combined_price)
        combined_price = processor.smart_data_cleaning(combined_price, 'Price')
        
        # 准备建模数据
        X_price_train, X_price_test, y_price_train = processor.prepare_final_features(
            combined_price, 'Price'
        )
        
        if (X_price_train is not None and y_price_train is not None and 
            len(X_price_train) > 0 and len(X_price_test) > 0):
            
            # 分割训练验证集 - 保持20%验证集比例
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_price_train, y_price_train, test_size=0.2, random_state=111,
                shuffle=True
            )
            
            print(f"📊 房价数据: 训练 {X_tr.shape}, 验证 {X_val.shape}, 测试 {X_price_test.shape}")
            
            # 使用基于非线性经验的AdvancedModelTrainer训练
            price_trainer = AdvancedModelTrainer(property_type='price')
            
            # 训练基于非线性经验的线性模型
            linear_results = price_trainer.train_linear_models_with_insights(X_tr, y_tr, X_val, y_val)
            
            # 选择最佳模型
            price_results = price_trainer.select_best_model(linear_results)
            all_results['Price'] = price_results
            
            # 预测测试集
            if price_trainer.best_model is not None:
                try:
                    price_preds = price_trainer.predict_with_insights(X_price_test)
                    test_ids = price_test['ID'] if 'ID' in price_test.columns else range(len(price_preds))
                    
                    for idx, pred in zip(test_ids, price_preds):
                        all_predictions.append({'ID': idx, 'Price': pred})
                    
                    print(f"✅ 房价测试集预测完成: {len(price_preds)} 条记录")
                    
                except Exception as e:
                    print(f"❌ 房价预测错误: {e}")
                    # 备用预测方法
                    try:
                        price_preds = price_trainer.best_model.predict(X_price_test)
                        price_preds = np.expm1(price_preds)
                        test_ids = price_test['ID'] if 'ID' in price_test.columns else range(len(price_preds))
                        
                        for idx, pred in zip(test_ids, price_preds):
                            all_predictions.append({'ID': idx, 'Price': pred})
                        print(f"✅ 房价测试集预测完成(备用方法): {len(price_preds)} 条记录")
                    except Exception as e2:
                        print(f"❌ 备用预测方法也失败: {e2}")
    
    # ===== 房租预测 =====  
    print("\n" + "="*60)
    print("🏠 房租建模（基于非线性经验优化）")
    print("="*60)
    
    combined_rent, rent_train, rent_test = processor.combine_datasets(
        rent_train_path, rent_test_path, 'Price'
    )
    
    if combined_rent is not None:
        # 数据预处理流程
        combined_rent = processor.comprehensive_feature_engineering(combined_rent)
        combined_rent = processor.smart_data_cleaning(combined_rent, 'Price')
        
        # 准备建模数据
        X_rent_train, X_rent_test, y_rent_train = processor.prepare_final_features(
            combined_rent, 'Price'
        )
        
        if (X_rent_train is not None and y_rent_train is not None and 
            len(X_rent_train) > 0 and len(X_rent_test) > 0):
            
            # 分割训练验证集 - 保持20%验证集比例
            X_tr_rent, X_val_rent, y_tr_rent, y_val_rent = train_test_split(
                X_rent_train, y_rent_train, test_size=0.2, random_state=111,
                shuffle=True
            )
            
            print(f"📊 房租数据: 训练 {X_tr_rent.shape}, 验证 {X_val_rent.shape}, 测试 {X_rent_test.shape}")
            
            # 使用基于非线性经验的AdvancedModelTrainer训练
            rent_trainer = AdvancedModelTrainer(property_type='rent')
            
            # 训练基于非线性经验的线性模型
            linear_results_rent = rent_trainer.train_linear_models_with_insights(X_tr_rent, y_tr_rent, X_val_rent, y_val_rent)
            
            # 选择最佳模型
            rent_results = rent_trainer.select_best_model(linear_results_rent)
            all_results['Rent'] = rent_results
            
            # 预测测试集
            if rent_trainer.best_model is not None:
                try:
                    rent_preds = rent_trainer.predict_with_insights(X_rent_test)
                    test_ids = rent_test['ID'] if 'ID' in rent_test.columns else range(len(rent_preds))
                    
                    for idx, pred in zip(test_ids, rent_preds):
                        all_predictions.append({'ID': idx, 'Price': pred})
                    
                    print(f"✅ 房租测试集预测完成: {len(rent_preds)} 条记录")
                    
                except Exception as e:
                    print(f"❌ 房租预测错误: {e}")
                    # 备用预测方法
                    try:
                        rent_preds = rent_trainer.best_model.predict(X_rent_test)
                        rent_preds = np.expm1(rent_preds)
                        test_ids = rent_test['ID'] if 'ID' in rent_test.columns else range(len(rent_preds))
                        
                        for idx, pred in zip(test_ids, rent_preds):
                            all_predictions.append({'ID': idx, 'Price': pred})
                        print(f"✅ 房租测试集预测完成(备用方法): {len(rent_preds)} 条记录")
                    except Exception as e2:
                        print(f"❌ 备用预测方法也失败: {e2}")
    
    # ===== 结果输出 =====
    print("\n" + "="*60)
    print("📊 最终结果")
    print("="*60)
    
    if all_predictions:
        # 保存预测结果
        submission_df = pd.DataFrame(all_predictions)
        submission_filename = 'enhanced_linear_submission_Class25Q2.csv'
        submission_df.to_csv(submission_filename, index=False)
        print(f"✅ 预测结果已保存: {submission_filename} ({len(submission_df)} 条记录)")
        
        # 保存详细结果
        for prop_type, results in all_results.items():
            if not results.empty:
                results_file = f'enhanced_linear_detailed_results_{prop_type}.csv'
                results.to_csv(results_file, index=False)
                print(f"📄 详细结果已保存: {results_file}")
                
                # 打印最佳模型结果
                best_model_row = results.loc[results['Out_sample_MAE'].idxmin()]
                print(f"\n🏆 {prop_type} 最佳模型:")
                print(f"   模型: {best_model_row['Model']}")
                print(f"   验证集MAE: {best_model_row['Out_sample_MAE']:,.2f}")
                print(f"   Kaggle分数: {best_model_row['Kaggle_Score']:.1f}")
                print(f"   使用特征数: {best_model_row['Features_Used']}")
                print(f"   非线性经验应用: {best_model_row.get('Nonlinear_Insights_Applied', False)}")
        
        # 汇总统计
        total_predictions = len(submission_df)
        price_predictions = len([p for p in all_predictions if p['Price'] > 100000])
        rent_predictions = total_predictions - price_predictions
        
        print(f"\n📈 汇总统计:")
        print(f"   总预测数: {total_predictions}")
        print(f"   房价预测: {price_predictions}")
        print(f"   房租预测: {rent_predictions}")
        
        # 性能提升分析
        print(f"\n🚀 基于非线性经验的改进:")
        print(f"   - 使用随机森林识别真正重要的特征")
        print(f"   - 应用非线性模型学到的特征交互模式")
        print(f"   - 智能特征变换建议")
        print(f"   - 优化的多项式特征和正则化")
        print(f"   - 改进的异常值处理策略")
        
    else:
        print("❌ 未生成任何预测结果")
    
    return all_results, all_predictions

# 运行改进的主流程
if __name__ == "__main__":
    results, predictions = run_enhanced_real_estate_pipeline()

=== 🏠 基于Untitled5非线性经验改进的房地产预测系统 ===


🏘️  房价建模（基于非线性经验优化）
✅ 成功加载: ruc_Class25Q2_train_price.csv (编码: UTF-8-SIG, 形状: (103871, 55))
✅ 成功加载: ruc_Class25Q2_test_price.csv (编码: UTF-8-SIG, 形状: (34017, 55))
📊 数据合并完成: 训练集 (103871, 56), 测试集 (34017, 56), 合并 (137888, 57)
🔧 开始综合特征工程...
🔧 开始高级特征工程...
✅ 特征工程完成，创建了 63 个新特征
✅ 空间聚类完成: 5 个聚类
✅ 特征工程完成: 形状 (137888, 121)
🧹 开始智能数据清洗...
🧹 移除目标缺失样本: 0 行
✅ 异常值处理: 移除 371 个样本, 剩余 103500 个样本
🔧 处理缺失值...
🚫 移除高缺失率特征: ['别墅类型', '抵押信息', '户型介绍', '环线位置', 'ID']
🧹 移除关键特征缺失行: 0 行
✅ 缺失值处理完成
✅ 数据清洗完成
🎯 准备最终特征...
训练集: (103500, 116), 测试集: (34017, 116)
移除零方差特征: ['有_客户反馈', '建筑结构_塔楼', '建筑结构_板楼', '建筑结构_塔板结合', '建筑结构_平房']
移除高度相关特征: ['区县', 'coord_x', 'coord_y', '供电_民电', '容积率_num', '停车位_num', '房屋用途_住宅']
✅ 最终特征: X_train (103500, 64), X_test (34017, 64)
📈 特征数量: 64
📊 房价数据: 训练 (82800, 64), 验证 (20700, 64), 测试 (34017, 64)
📈 基于非线性经验训练改进的线性模型...
🔍 从非线性模型提取关键经验...
✅ 提取到 30 个关键特征
📊 前5重要特征: ['城市', '建筑面积_数值', 'lon', '面积_log', 'lat']
✅ OLS_Enhanced: 训练MAE = 627,809.12, 验证MAE = 631,630.11
✅ Ridge_E

In [22]:
# Cell 6: 改进的模型解释和深入分析
def perform_deep_analysis(processor, trainer, X_train, y_train, X_test, property_type):
    """执行深入的模型分析 - 基于Untitled5经验改进"""
    print(f"\n🔍 执行 {property_type} 深度分析...")
    
    # 1. 特征相关性分析
    plt.figure(figsize=(15, 12))
    
    # 使用重要特征进行相关性分析
    if hasattr(trainer, 'important_features') and trainer.important_features:
        analysis_features = trainer.important_features[:15]  # 只分析前15个重要特征
    else:
        # 选择数值型特征进行相关性分析
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 20:
            rf = RandomForestRegressor(n_estimators=50, random_state=111)
            rf.fit(X_train[numeric_cols], y_train)
            importances = pd.DataFrame({
                'feature': numeric_cols,
                'importance': rf.feature_importances_
            }).sort_values('importance', ascending=False)
            analysis_features = importances.head(15)['feature'].tolist()
        else:
            analysis_features = numeric_cols
    
    corr_matrix = X_train[analysis_features].corr()
    
    # 绘制相关性热力图
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
    plt.title(f'{property_type} - 重要特征相关性热力图')
    plt.tight_layout()
    plt.show()
    
    # 2. 目标变量分布分析
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    original_prices = np.expm1(y_train)
    plt.hist(original_prices, bins=50, alpha=0.7, edgecolor='black')
    plt.title(f'{property_type} - 原始价格分布')
    plt.xlabel('价格')
    plt.ylabel('频数')
    
    plt.subplot(1, 2, 2)
    plt.hist(y_train, bins=50, alpha=0.7, edgecolor='black')
    plt.title(f'{property_type} - 对数价格分布')
    plt.xlabel('对数价格')
    plt.ylabel('频数')
    
    plt.tight_layout()
    plt.show()
    
    # 3. 关键特征与价格关系
    if len(analysis_features) >= 3:
        key_features = analysis_features[:3]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        for i, feature in enumerate(key_features):
            axes[i].scatter(X_train[feature], np.expm1(y_train), alpha=0.3, s=10)
            
            # 添加趋势线
            z = np.polyfit(X_train[feature], np.expm1(y_train), 1)
            p = np.poly1d(z)
            x_range = np.linspace(X_train[feature].min(), X_train[feature].max(), 100)
            axes[i].plot(x_range, p(x_range), "r--", alpha=0.8)
            
            axes[i].set_xlabel(feature)
            axes[i].set_ylabel('价格')
            axes[i].set_title(f'{property_type} - {feature} vs 价格')
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # 4. 模型性能分析（如果可用）
    if trainer.best_model is not None:
        # 残差分析
        if hasattr(trainer, 'important_features') and trainer.important_features:
            X_analysis = X_train[trainer.important_features]
        else:
            X_analysis = X_train
            
        y_pred_log = trainer.best_model.predict(X_analysis)
        y_true = np.expm1(y_train)
        y_pred = np.expm1(y_pred_log)
        
        residuals = y_true - y_pred
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 残差分布
        axes[0, 0].hist(residuals, bins=50, alpha=0.7, edgecolor='black')
        axes[0, 0].axvline(0, color='red', linestyle='--')
        axes[0, 0].set_title(f'{trainer.best_model_name} - 残差分布')
        axes[0, 0].set_xlabel('残差')
        axes[0, 0].set_ylabel('频数')
        
        # 残差vs预测值
        axes[0, 1].scatter(y_pred, residuals, alpha=0.5, s=10)
        axes[0, 1].axhline(0, color='red', linestyle='--')
        axes[0, 1].set_title(f'{trainer.best_model_name} - 残差vs预测值')
        axes[0, 1].set_xlabel('预测值')
        axes[0, 1].set_ylabel('残差')
        
        # 预测值vs真实值
        axes[1, 0].scatter(y_true, y_pred, alpha=0.5, s=10)
        max_val = max(y_true.max(), y_pred.max())
        axes[1, 0].plot([0, max_val], [0, max_val], 'r--')
        axes[1, 0].set_title(f'{trainer.best_model_name} - 预测值vs真实值')
        axes[1, 0].set_xlabel('真实值')
        axes[1, 0].set_ylabel('预测值')
        
        # QQ图
        stats.probplot(residuals, dist="norm", plot=axes[1, 1])
        axes[1, 1].set_title(f'{trainer.best_model_name} - QQ图')
        
        plt.tight_layout()
        plt.show()
        
        # 打印统计信息
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        print(f"📊 {trainer.best_model_name} 残差分析:")
        print(f"   MAE: {mae:,.2f}")
        print(f"   R²: {r2:.4f}")
        print(f"   残差均值: {residuals.mean():.2f}")
        print(f"   残差标准差: {residuals.std():.2f}")
    
    print(f"✅ {property_type} 深度分析完成")

# 在主流程运行后调用深度分析的函数
def run_analysis_after_main():
    """在主流程运行后执行深度分析"""
    print("\n" + "="*60)
    print("🔍 启动深度分析")
    print("="*60)
    
    # 检查房价分析变量是否存在
    try:
        # 这些变量应该在主流程中定义
        if 'price_trainer' in globals() and 'X_price_train' in globals() and 'y_price_train' in globals() and 'X_price_test' in globals():
            print("📊 开始房价深度分析...")
            perform_deep_analysis(processor, price_trainer, X_price_train, y_price_train, X_price_test, 'Price')
        else:
            print("⚠️ 房价分析变量未找到，跳过房价分析")
    except Exception as e:
        print(f"⚠️ 房价分析错误: {e}")
    
    # 检查房租分析变量是否存在
    try:
        if 'rent_trainer' in globals() and 'X_rent_train' in globals() and 'y_rent_train' in globals() and 'X_rent_test' in globals():
            print("📊 开始房租深度分析...")
            perform_deep_analysis(processor, rent_trainer, X_rent_train, y_rent_train, X_rent_test, 'Rent')
        else:
            print("⚠️ 房租分析变量未找到，跳过房租分析")
    except Exception as e:
        print(f"⚠️ 房租分析错误: {e}")

# 注意：要在主流程运行后手动调用这个函数
       